**Quick setup**: Install dependencies if needed (run in the notebook or environment).

Run this once in your environment:

In [ ]:
# If running in a fresh environment, install required packages (uncomment to run).
# Note: prefer using your conda env from `environment_selfies.yml` for RDKit if needed.
# !pip install -q transformers datasets evaluate scikit-learn accelerate

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('libs imported')

In [ ]:
# Load the pre-split CSVs produced by `src/build_hf_dataset.py`
data_files = {
    'train': 'data/hf_dataset_small_train.csv',
    'validation': 'data/hf_dataset_small_val.csv',
    'test': 'data/hf_dataset_small_test.csv',
}
dataset = load_dataset('csv', data_files=data_files)
dataset


In [ ]:
# Inspect a few examples
for split in dataset:
    print(split, len(dataset[split]))
    print(dataset[split][0])


## Tokenization
We use a standard tokenizer (`distilbert-base-uncased`) for this PoC. For SELFIES-specific modeling, replace with a tokenizer trained for SELFIES tokens or a character-level tokenizer consistent with your pretrained model.

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized

## Model (regression)
We load a pretrained encoder and attach a regression head (num_labels=1).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
# Ensure the model is configured for regression
model.config.problem_type = 'regression'

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer)

print('model and data collator ready')

In [ ]:
# Compute metrics for regression
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.squeeze(preds)
    mse = mean_squared_error(labels, preds)
    return {
        'mse': float(mse),
        'rmse': float(np.sqrt(mse)),
        'mae': float(mean_absolute_error(labels, preds)),
        'r2': float(r2_score(labels, preds)),
    }

# Training args: small and fast for PoC
training_args = TrainingArguments(
    output_dir='./results_poc',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    evaluation_strategy='epoch',
    save_strategy='no',
    logging_steps=1,
    learning_rate=5e-5,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('trainer ready')

## Train (PoC)
Run the training loop. With the tiny dataset this will quickly overfit — expect meaningless metrics, but it verifies the pipeline.

In [ ]:
train_results = trainer.train()
print(train_results)


In [ ]:
# Evaluate on the test split
metrics = trainer.evaluate(eval_dataset=tokenized['test'])
print('Test metrics:', metrics)


## Save the model
Save the finetuned model locally for quick inference tests.

In [ ]:
os.makedirs('models/poc_distilbert_regressor', exist_ok=True)
trainer.save_model('models/poc_distilbert_regressor')
print('model saved to models/poc_distilbert_regressor')

## Next steps / Notes
- Replace `distilbert-base-uncased` with a SELFIES-aware model/tokenizer if available (recommended).
- Increase dataset size (convert `data/dv_values_by_cid.csv` SMILES → SELFIES) before meaningful training.
- Use `transformers` training optimizations (FP16, gradient accumulation, `accelerate`) for larger runs.